# Convergence across recurrent depth

Do each layer's residual stream and attention pattern stabilise across recurrences?
For each of six layerwise measurements, this notebook produces three full-width
figures: **main** and **appendix** each have four models in one row; **all** stacks
those rows. The ordered groups come from `experiments/paper_models.py`.

The main group is Ouro 1.4B, Huginn-0125, Retrofitted Llama, and Parcae 770M.
The appendix contains Retrofitted OLMo, Retrofitted TinyLlama, Parcae 140M, and
Parcae 370M. Each model is collected once and reused across the figures.

All models run for 128 recurrences by default. `mode="quick"` uses one seeded
example and saves figures only in the run directory. `mode="full"` defaults to
256 examples and also updates the paper figures. SLURM selects full mode; an
explicit `limit` overrides either sample count.

In [ ]:
mode = "quick"  # "quick" saves locally; "full" also updates paper figures.
dataset = "gsm8k"  # "gsm8k" or "hellaswag"
recurrences = 128  # Applied to every model in the shared paper groups.
model_recurrences = None  # Optional explicit [(model, recurrences), ...] subset.
limit = None  # Default: 1 (quick), 256 (full); 0 means the entire split.
seed = 42
device = "auto"
dtype = "auto"
output_dir = None

cache_dir = None  # Defaults to LOOPED_INTERNALS_CACHE_DIR or /data-gauss/hugbla/looped-internals-cache.
cache_mode = None  # Environment default; "reuse", "require" (no inference), or "off".
analysis_version = 1  # Bump when this notebook's scientific reduction changes.
source_run = None  # Optional completed run directory with matching scientific settings.
cache_profile_pairs = []  # Optional [(model, recurrences)] also needed by stages.

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from dataclasses import asdict
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
import torch

from experiments.cache import ExperimentCache
from experiments.paper_models import paper_model_pairs
from experiments.saved_runs import SavedRun
from experiments.support import (
    COLORMAPS,
    MODEL_LABELS,
    add_depth_colorbar,
    collect_metrics,
    configure_experiment,
    export_paper_figures,
    model_subplots,
    save_figure,
    save_results,
)
from looped_internals.analysis import (
    compare_residuals,
    compute_attn_pattern_diff_norm,
)
from looped_internals.capture.extraction import _cached_virtual_layers

plt.style.use(ROOT / "plots" / "matlab.mplstyle")

# Resolve shared defaults after Papermill overrides; collect repeated panels once.
model_recurrences, figure_pairs = paper_model_pairs(recurrences, model_recurrences)

sample_limit = limit if limit is not None else (1 if mode == "quick" else 256)

configs = {
    (name, recurrences): configure_experiment(
        experiment="convergence",
        dataset=dataset,
        mode=mode,
        models=[name],
        num_steps=recurrences,
        limit=sample_limit,
        seed=seed,
        device=device,
        dtype=dtype,
        output_dir=output_dir,
    )
    for name, recurrences in model_recurrences
}
config = next(iter(configs.values()))

result_cache = ExperimentCache(cache_dir, cache_mode)
cache_profile_pairs = {tuple(pair) for pair in cache_profile_pairs}
if cache_profile_pairs - configs.keys():
    raise ValueError("cache_profile_pairs must be a subset of model_recurrences")

In [ ]:
METRICS = {
    "to_final_state_residual_diff_norm": "Final-state norm",
    "to_final_state_residual_cosine_sim": "Final-state cosine similarity",
    "to_final_state_attn_pattern_diff_norm": "Final-state attn. diff. norm",
    "successive_recurrence_residual_diff_norm": "Difference norm",
    "successive_recurrence_residual_cosine_sim": "Successive cosine similarity",
    "successive_recurrence_attn_pattern_diff_norm": "Successive attn. diff. norm",
}


def recurrent_metric_names(structure):
    """Capture the recurrent residuals and attention used by these curves."""
    names = {
        f"{entry.hook_prefix}.{suffix}"
        for entry in structure.virtual_layers(1)
        if entry.iteration is not None
        for suffix in ("hook_out", "attn.hook_pattern")
    }
    return names.__contains__


def reduce_convergence_cache(cache, input_ids, structure):
    """Average tokens or heads after each same-layer comparison."""
    # Accumulation also gives single-pass prelude/coda hooks an .iter_0 suffix.
    recurrent_layers = {
        entry.layer_index
        for entry in structure.virtual_layers(1)
        if entry.iteration is not None
    }
    by_layer = {}
    for entry in _cached_virtual_layers(cache, structure):
        if entry.layer_index in recurrent_layers:
            by_layer.setdefault(entry.layer_index, []).append(entry)
    if not by_layer or any(len(entries) < 2 for entries in by_layer.values()):
        raise ValueError("Every recurrent layer needs at least two observations")

    curves = {metric: [] for metric in METRICS}
    for entries in by_layer.values():
        # One final reference per physical layer, including the self-comparison.
        final_residual = cache.cache_dict[entries[-1].residual_key].float()
        final_attention = cache.cache_dict[entries[-1].attn_key]
        values = {metric: [] for metric in METRICS}
        previous = None
        for entry in entries:
            residual = cache.cache_dict[entry.residual_key].float()
            attention = cache.cache_dict[entry.attn_key]
            references = {"to_final_state": (final_residual, final_attention)}
            if previous is not None:
                references["successive_recurrence"] = previous
            for family, (reference_residual, reference_attention) in references.items():
                comparison = compare_residuals(residual, reference_residual)
                values[f"{family}_residual_cosine_sim"].append(
                    comparison.cosine_sim.mean(-1)
                )
                values[f"{family}_residual_diff_norm"].append(
                    comparison.diff_norm.mean(-1)
                )
                values[f"{family}_attn_pattern_diff_norm"].append(
                    compute_attn_pattern_diff_norm(attention, reference_attention).mean(
                        -1
                    )
                )
            previous = residual, attention
        for metric in METRICS:
            curves[metric].append(torch.stack(values[metric], dim=-1).cpu())

    return [
        {
            "input_ids": input_ids[batch : batch + 1].cpu(),
            "summary": {
                "layers": list(by_layer),
                "curves": {
                    metric: [layer[batch].tolist() for layer in values]
                    for metric, values in curves.items()
                },
            },
        }
        for batch in range(len(input_ids))
    ]

## Generate and reduce

Each pair runs on the same seeded examples. The cache retains only recurrent-layer
residuals and attention patterns. Reduce them directly to the same-layer final and
successive curves used below, averaging tokens or heads within each example before
releasing the cache. No cross-layer comparison tensors or unrelated stage metrics
are constructed. Layer metadata identifies the recurrent block, so prelude and coda
lengths are not hard-coded.

Save the curves and run metadata in `overview.pt` in a new run directory. Once this
cell finishes, the figure cells can be rerun without repeating inference.
When cache population also requests matching stages profiles, `cache_profile_pairs`
adds their layer measurements to the same forward pass. These profiles include the
prelude and coda and are stored separately; they do not change the convergence curves.


In [ ]:
settings = asdict(config)
del settings["models"], settings["num_steps"]
settings.update(
    model_recurrences=model_recurrences,
    figure_pairs=figure_pairs,
    output_dir=str(config.output_dir),
)
settings["versions"] = {
    name: version(name) for name in ("torch", "transformer-lens", "transformers")
}
settings["cache"] = result_cache.settings
previous = SavedRun(source_run, settings, analysis_version=analysis_version)
previous.check_pairs(settings)
settings.update(analysis_version=analysis_version, source_run=source_run)
settings["cache"]["profile_pairs"] = sorted(cache_profile_pairs)
run_dir = save_results({}, settings, config.output_dir)

summaries, run_info = {}, {}
for pair, pair_config in configs.items():
    if pair in previous:
        summaries[pair] = previous.data["summaries"][pair]
        run_info[pair] = previous.info(pair)
        continue
    name, recurrences = pair
    results, info = collect_metrics(
        pair_config,
        None,
        cache=result_cache,
        analysis_version=analysis_version,
        cache_reducer=reduce_convergence_cache,
        names_filter_factory=recurrent_metric_names,
        save_profiles=pair in cache_profile_pairs,
    )
    summaries[pair] = results[name]
    run_info[pair] = info
    del results

torch.save(
    {"summaries": summaries, "run": run_info, "figure_pairs": figure_pairs},
    run_dir / "overview.pt",
)
result_cache.write_manifest(run_dir)

## What the curves measure

Let $h_{r,l,t}$ be the residual stream after recurrence $r$, layer $l$, and token
$t$, and let $A_{r,l,a}$ be the attention pattern for head $a$.
Every comparison below uses **the same layer** on both sides.

| Measurement | Residual stream | Attention |
| --- | --- | --- |
| To final recurrence $R$ | $\|h_{r,l,t}-h_{R,l,t}\|_2$ and cosine similarity | $\|A_{r,l,a}-A_{R,l,a}\|_F$ |
| Between successive recurrences | $\|h_{r,l,t}-h_{r-1,l,t}\|_2$ and cosine similarity | $\|A_{r,l,a}-A_{r-1,l,a}\|_F$ |

Average tokens (or attention heads) **within each example first**, then average
examples, giving each example equal weight. Inputs contain both the context and
the likelihood task's continuation. Each coloured line is one recurrent layer;
shading is one sample standard deviation across examples, or zero for one example.
Norms are unnormalised, so their scales can differ between models.

The final observed state is a finite-rollout reference, not a proven fixed point.
As in the paper plots, we include recurrence $R$: its self-distance is zero and
its self-cosine is one for nonzero residuals by construction. Successive comparisons
start at recurrence 2 and help distinguish stability from agreement with this reference.

In [ ]:
def plot_convergence(metric, filename, *, group):
    """Plot one paper group, reusing the collected layerwise curves."""
    pairs = figure_pairs[group]
    if not pairs:
        return  # An explicit subset can omit a whole group.
    fig, axes = model_subplots(len(pairs))
    cmap = plt.colormaps[COLORMAPS["layers"]]
    for ax, pair in zip(axes, pairs, strict=True):
        name, recurrences = pair
        examples = summaries[pair]
        values = np.asarray([example["curves"][metric] for example in examples])
        mean = values.mean(axis=0)
        std = values.std(axis=0, ddof=1 if len(examples) > 1 else 0)
        first = 1 if metric.startswith("to_final_state_") else 2
        x = np.arange(first, recurrences + 1)
        colors = cmap(np.linspace(0, 1, len(mean)))
        for layer, average, spread, color in zip(
            examples[0]["layers"], mean, std, colors, strict=True
        ):
            ax.plot(x, average, color=color, label=f"Layer {layer}")
            ax.fill_between(
                x, average - spread, average + spread, color=color, alpha=0.2
            )
        ax.set(
            title=MODEL_LABELS[name].replace("Retrofitted ", "R."), xlabel="Recurrence"
        )
        if ax.get_subplotspec().is_first_col():
            ax.set_ylabel(METRICS[metric])
    add_depth_colorbar(fig, axes, "layer")
    save_figure(fig, run_dir, f"{filename}_{group}", bbox_inches=None)

## Distance to each layer's final state

Residual norm, residual cosine similarity, and attention-pattern distance.
Each metric has a main, appendix, and combined figure at the full paper width.

In [ ]:
plot_convergence(
    "to_final_state_residual_diff_norm",
    "fixed_point_norms",
    group="main",
)

In [ ]:
plot_convergence(
    "to_final_state_residual_diff_norm",
    "fixed_point_norms",
    group="appendix",
)

In [ ]:
plot_convergence(
    "to_final_state_residual_diff_norm",
    "fixed_point_norms",
    group="all",
)

In [ ]:
plot_convergence(
    "to_final_state_residual_cosine_sim",
    "fixed_point_cosine_similarities",
    group="main",
)

In [ ]:
plot_convergence(
    "to_final_state_residual_cosine_sim",
    "fixed_point_cosine_similarities",
    group="appendix",
)

In [ ]:
plot_convergence(
    "to_final_state_residual_cosine_sim",
    "fixed_point_cosine_similarities",
    group="all",
)

In [ ]:
plot_convergence(
    "to_final_state_attn_pattern_diff_norm",
    "fixed_point_attention_differences",
    group="main",
)

In [ ]:
plot_convergence(
    "to_final_state_attn_pattern_diff_norm",
    "fixed_point_attention_differences",
    group="appendix",
)

In [ ]:
plot_convergence(
    "to_final_state_attn_pattern_diff_norm",
    "fixed_point_attention_differences",
    group="all",
)

## Changes between successive recurrences

Distances approaching zero and cosine similarity approaching one indicate that
successive applications of the recurrent block change the layer's state less.

In [ ]:
plot_convergence(
    "successive_recurrence_residual_diff_norm",
    "successive_recurrence_residual_diff_norm",
    group="main",
)

In [ ]:
plot_convergence(
    "successive_recurrence_residual_diff_norm",
    "successive_recurrence_residual_diff_norm",
    group="appendix",
)

In [ ]:
plot_convergence(
    "successive_recurrence_residual_diff_norm",
    "successive_recurrence_residual_diff_norm",
    group="all",
)

In [ ]:
plot_convergence(
    "successive_recurrence_residual_cosine_sim",
    "successive_recurrence_residual_cosine_sim",
    group="main",
)

In [ ]:
plot_convergence(
    "successive_recurrence_residual_cosine_sim",
    "successive_recurrence_residual_cosine_sim",
    group="appendix",
)

In [ ]:
plot_convergence(
    "successive_recurrence_residual_cosine_sim",
    "successive_recurrence_residual_cosine_sim",
    group="all",
)

In [ ]:
plot_convergence(
    "successive_recurrence_attn_pattern_diff_norm",
    "successive_recurrence_attn_pattern_diff_norm",
    group="main",
)

In [ ]:
plot_convergence(
    "successive_recurrence_attn_pattern_diff_norm",
    "successive_recurrence_attn_pattern_diff_norm",
    group="appendix",
)

In [ ]:
plot_convergence(
    "successive_recurrence_attn_pattern_diff_norm",
    "successive_recurrence_attn_pattern_diff_norm",
    group="all",
)

In [ ]:
print(run_dir)
paper_dir = export_paper_figures(config, run_dir, dataset=config.dataset)
if paper_dir is not None:
    print(f"Saved paper figures to {paper_dir}")